In [5]:
pip install osmnx

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install overpy

  Obtaining dependency information for overpy from https://files.pythonhosted.org/packages/ec/12/315d56e8386a4060d9a978a34ad48a9af072b67f40504eaa2f44197a15e5/overpy-0.7-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.


In [6]:
import requests
import geopandas as gpd
from shapely.geometry import Point

query = """
[out:json][timeout:120];
(
  node["amenity"~"pharmacy|restaurant|cafe|bar|fast_food|supermarket|bank"](39.40,-0.42,39.55,-0.30);
  node["shop"~"bakery|clothes|hairdresser|convenience"](39.40,-0.42,39.55,-0.30);
);
out body;
"""

headers = {
    "User-Agent": "Python/requests OSM data download academic project",
    "Accept": "application/json, text/json, */*",
    "Accept-Encoding": "gzip, deflate",
}

print("Descargando todos los locales de Valencia...")
response = requests.get(
    "https://overpass-api.de/api/interpreter",
    params={"data": query},
    headers=headers,
    timeout=120
)
response.raise_for_status()
nodos = response.json()["elements"]
print(f"  -> {len(nodos)} nodos encontrados")

def get_tipo(tags):
    amenity = tags.get("amenity", "")
    shop    = tags.get("shop", "")
    if amenity in {"pharmacy","restaurant","cafe","bar","fast_food","supermarket","bank"}:
        return amenity
    if shop in {"bakery","clothes","hairdresser","convenience"}:
        return shop
    return "otro"

registros = []
for nodo in nodos:
    tags = nodo.get("tags", {})
    registros.append({
        "id":       nodo["id"],
        "lat":      nodo["lat"],
        "lon":      nodo["lon"],
        "nombre":   tags.get("name", "Sin nombre"),
        "tipo":     get_tipo(tags),
        "geometry": Point(nodo["lon"], nodo["lat"])
    })

print(f"Total locales: {len(registros)}")

gdf = gpd.GeoDataFrame(registros, crs="EPSG:4326")
gdf.to_file("locales_valencia.json", driver="GeoJSON")

print("Guardado en locales_valencia.json")
print(gdf["tipo"].value_counts())

Descargando todos los locales de Valencia...
  -> 3377 nodos encontrados
Total locales: 3377
Guardado en locales_valencia.json
tipo
restaurant     1223
cafe            418
pharmacy        314
bar             265
clothes         249
bank            222
fast_food       215
hairdresser     195
bakery          184
convenience      90
otro              2
Name: count, dtype: int64
